In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
train = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
test = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")

In [7]:
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [8]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [9]:
train.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

I FILLED THE MISSING VALUES OF AGE USING THE MEDIAN:

In [11]:
train["Age"] = train["Age"].fillna(train["Age"].median())

In [12]:
train["Age"].isnull().sum()

np.int64(0)

FILLED THE MISSING VALUES IN EMBARKED

In [13]:
train["Embarked"] = train["Embarked"].fillna(train["Embarked"].mode()[0])

In [14]:
train["Embarked"].isnull().sum()

np.int64(0)

 I Removed the Cabin column because it has too many missing values.

In [15]:
train = train.drop("Cabin", axis=1)

In [16]:
train.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Embarked'],
      dtype='object')

I Converted the Sex column from text to the numbers

In [17]:
train["Sex"] = train["Sex"].map({
    "male": 0,
    "female": 1
})

In [18]:
train["Sex"].head()

0    0
1    1
2    1
3    1
4    0
Name: Sex, dtype: int64

I Converted the Embarked column from text to numbers.

In [19]:
train["Embarked"] = train["Embarked"].map({
    "S": 0,
    "C": 1,
    "Q": 2
})

In [20]:
train["Embarked"].head()

0    0
1    1
2    0
3    0
4    0
Name: Embarked, dtype: int64

 I selected only the important features input columns and target column.

In [22]:
features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
X = train[features]
y = train["Survived"]

In [23]:
X.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,0,22.0,1,0,7.2500,0
1,1,1,38.0,1,0,71.2833,1
2,3,1,26.0,0,0,7.9250,0
3,1,1,35.0,1,0,53.1000,0
4,3,0,35.0,0,0,8.0500,0


In this I splited the data into training and testing sets.

In [24]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [25]:
X_train.shape, X_val.shape

((712, 7), (179, 7))

In this step I used Random Forest Classifier to train.

In [27]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)

model.fit(X_train, y_train)
print("Model Trained Successfully")

Model Trained Successfully


Predictions on the Test data.

In [29]:
y_pred = model.predict(X_val)

In [30]:
print(y_pred[:10])

[0 0 0 1 0 1 1 0 1 1]


Checking the Accuracy of the model

In [31]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_val, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.8268156424581006


I used ipywidgets for neat Visualization and Understanding

In [35]:
import ipywidgets as widgets
from IPython.display import display

pclass = widgets.Dropdown(options=[1, 2, 3], description='Pclass')
sex = widgets.Dropdown(options=['male', 'female'], description='Sex')
age = widgets.FloatText(description='Age')
sibsp = widgets.IntText(description='SibSp')
parch = widgets.IntText(description='Parch')
fare = widgets.FloatText(description='Fare')
embarked = widgets.Dropdown(options=['S', 'C', 'Q'], description='Embarked')

button = widgets.Button(description='Predict')
output = widgets.Output()

def predict(b):
    output.clear_output()

    sex_val = 0 if sex.value == 'male' else 1
    emb_val = {'S': 0, 'C': 1, 'Q': 2}[embarked.value]

    data = [[
        pclass.value,
        sex_val,
        age.value,
        sibsp.value,
        parch.value,
        fare.value,
        emb_val
    ]]

    pred = model.predict(data)[0]

    with output:
        if pred == 1:
            print("✅ Survived")
        else:
            print("❌ Did Not Survive")

button.on_click(predict)

display(
    pclass,
    sex,
    age,
    sibsp,
    parch,
    fare,
    embarked,
    button,
    output
)

Dropdown(description='Pclass', options=(1, 2, 3), value=1)

Dropdown(description='Sex', options=('male', 'female'), value='male')

FloatText(value=0.0, description='Age')

IntText(value=0, description='SibSp')

IntText(value=0, description='Parch')

FloatText(value=0.0, description='Fare')

Dropdown(description='Embarked', options=('S', 'C', 'Q'), value='S')

Button(description='Predict', style=ButtonStyle())

Output()